In [1]:
!pip install -q surya-ocr pdf2image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 22.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 117.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 124.0 MB/s eta 0:00:0000:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
goog

In [2]:
!nvidia-smi

Fri Jun 19 12:19:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [20]:
!surya_ocr "/content/BPHS_hindi_part_1.pdf" --page_range 10 --output_dir "/content/output" 

In [3]:
# Get a CUDA-enabled llama.cpp build that matches Colab's Ubuntu 22.04 + NVIDIA GPU
!URL=$(curl -s https://api.github.com/repos/ai-dock/llama.cpp-cuda/releases/latest | grep "browser_download_url.*cuda-12.8-amd64.tar.gz" | cut -d '"' -f4) && \
 echo "Downloading: $URL" && \
 wget -q "$URL" -O /content/llama_cuda.tar.gz && \
 mkdir -p /content/llama_cpp_cuda && \
 tar -xzf /content/llama_cuda.tar.gz -C /content/llama_cpp_cuda && \
 find /content/llama_cpp_cuda -name "llama-server"

Downloading: https://github.com/ai-dock/llama.cpp-cuda/releases/download/b9716/llama.cpp-b9716-cuda-12.8-amd64.tar.gz
/content/llama_cpp_cuda/cuda-12.8/llama-server


In [4]:
import os

os.environ["SURYA_INFERENCE_BACKEND"] = "llamacpp"
os.environ["LLAMA_CPP_BINARY"] = "/content/llama_cpp_cuda/cuda-12.8/llama-server"
os.environ["LD_LIBRARY_PATH"] = "/content/llama_cpp_cuda/cuda-12.8:" + os.environ.get("LD_LIBRARY_PATH", "")

In [7]:
!ls -R /content/output

ls: cannot access '/content/output': No such file or directory


In [18]:
import json
import re
from html import unescape

with open("/content/output_1/BPHS_hindi_part_1/results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

def clean_html(html):
    text = re.sub(r"<[^>]+>", " ", html)
    text = unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for page in data.get("pages", []):
    print(f"\n--- PAGE {page.get('page_num', '')} ---\n")
    for block in page.get("blocks", []):
        if "html" in block:
            print(clean_html(block["html"]))
        elif "text" in block:
            print(block["text"])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
shutil.copy('/content/output/clean_text.txt', '/content/drive/MyDrive/clean_text.txt')

In [6]:
import os
os.environ["SURYA_INFERENCE_BACKEND"] = "llamacpp"
os.environ["LLAMA_CPP_BINARY"] = "/content/llama_cpp_cuda/cuda-12.8/llama-server"
os.environ["LD_LIBRARY_PATH"] = "/content/llama_cpp_cuda/cuda-12.8:" + os.environ.get("LD_LIBRARY_PATH", "")

In [ ]:
!surya_ocr "/content/BPHS_hindi_part_1.pdf" --page_range 150-200 --output_dir "/content/output_2"

2026-06-19 14:24:45,610 [INFO] surya: Downloading surya-2.gguf and surya-2-mmproj.gguf from datalab-to/surya-ocr-2-gguf
2026-06-19 14:24:45,823 [INFO] surya: llama-server ctx-size=98304 (~12288/slot × 8 parallel slots)
2026-06-19 14:24:45,824 [INFO] surya: Spawning llamacpp server on port 51931
2026-06-19 14:24:45,824 [INFO] surya: Spawning: /content/llama_cpp_cuda/cuda-12.8/llama-server -m /root/.cache/huggingface/hub/models--datalab-to--surya-ocr-2-gguf/snapshots/6a3a4c30e5e74446d4f8b6afd05b2f2da970f470/surya-2.gguf --mmproj /root/.cache/huggingface/hub/models--datalab-to--surya-ocr-2-gguf/snapshots/6a3a4c30e5e74446d4f8b6afd05b2f2da970f470/surya-2-mmproj.gguf -ngl 99 --host 127.0.0.1 --port 51931 --parallel 8 --ctx-size 98304 --alias datalab-to/surya-ocr-2 --jinja
2026-06-19 14:24:53,407 [INFO] surya: llamacpp server ready on port 51931 (model=datalab-to/surya-ocr-2)


In [ ]:
!find /content/output_2/BPHS_hindi_part_1 -type f

/content/output_2/BPHS_hindi_part_1/results.json


In [ ]:
import json
import re
from html import unescape

with open("/content/output_2/BPHS_hindi_part_1/results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

book_key = list(data.keys())[0]
pages = data[book_key]

def clean_html(html):
    text = re.sub(r"<br\s*/?>", "\n", html)
    text = re.sub(r"</p>", "\n\n", text)
    text = re.sub(r"</h1>|</h2>|</h3>", "\n\n", text)
    text = re.sub(r"</tr>", "\n", text)
    text = re.sub(r"</td>|</th>", " | ", text)

    text = re.sub(r"<[^>]+>", "", text)

    text = unescape(text)

    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

output_file = "/content/clean_text.md"

with open(output_file, "w", encoding="utf-8") as out:
    for page in pages:

        out.write(f"\n\n# PAGE {page['page']}\n\n")

        blocks = sorted(
            page["blocks"],
            key=lambda x: x.get("reading_order", 9999)
        )

        for block in blocks:
            cleaned = clean_html(block.get("html", ""))

            if cleaned:
                out.write(cleaned + "\n")

print("Saved:", output_file)

Saved: /content/clean_text.md


In [42]:
!head -100 /content/clean_text.md



# PAGE 1

द्वादशभावविचाराध्यायः
१५३
शुक्रो बुधोऽथवा जीवो लग्ने चन्द्रसमन्वितः।
लग्नात्केन्द्रगतो वापि राजलक्षणसंयुतः।।२८।।
यदि शुक्र, बुध वा गुरु चन्द्रमा से युत होकर लग्न से केन्द्र में गये हों तो बालक राजलक्षण से युक्त होता है।।२८।।
रविचन्द्रौ च ह्येकस्थावेकांशकसमन्वितौ।
त्रिमात्रं च त्रिभिर्मासैर्भ्रात्रा पित्रा च जीवति।।२९।।
रवि-चन्द्रमा एक स्थान में एक ही अंश में हों तो बालक को तीन मास के अन्दर तीन माताएँ होती हैं और भाई पिता से जीवित रहता है।।२९।।
लग्ने राहुसमायुक्ते तथा सोमनिरीक्षिते।
लग्नांशे मन्दसूरी चेज्जातश्च यमलो भवेत्।।३०।।
लग्न में राहु हो, उसे चन्द्रमा देखता हो तथा लग्न के नवमांश में शनि-गुरु हों तो यमल बालक होते हैं।।३०।।
अथ धनभावफलम्—
शुक्रेण युक्तो यदि नेत्रनाथः शुक्रस्य स्वोच्चांशगृहे गतो वा।
सम्बन्धवान्स्याद्यदि देहपेन नेत्रं विधत्ते विपरीतभावम्।।३१।।
यदि धन भाव का स्वामी शुक्र से युक्त हो और शुक्र के उच्च, नवांश, गृह में हो और लग्नेश से संबंध करता हो तो टेढ़े नेत्र होते हैं।।३१।।
तत्र स्थितौ चन्द्ररवी निशान्धं जात्यन्धतां नेत्रपदेहपार्काः।
पैत्रर्क्षनाथेन युतास्त

In [ ]:
import os

drive_folder = "/content/drive/MyDrive/BPHS_Hindi"
os.makedirs(drive_folder, exist_ok=True)

print("Created:", drive_folder)

Created: /content/drive/MyDrive/BPHS_Hindi


In [43]:
import shutil

shutil.copy(
    "/content/clean_text.md",
    "/content/drive/MyDrive/BPHS_Hindi/BPHS_part1_100_149.md"
)

print("Saved to Drive")

Saved to Drive
